In [0]:
%sql
create schema if not exists dbcertification_rev.demo_opt;

In [0]:
# Skewed key 80% of rows land on key = 1
from pyspark.sql.functions import *

df = (spark.range(0, 20000000)
      .withColumn("customer_id", when(col("id") % 10 < 8, lit(1)).otherwise((col("id") % 500) + 2))
      .withColumn("amount", (rand() * 1000).cast("double"))
      .withColumn("region", expr("array('east','west','north','south')[int(rand()*4)]"))   
      )

# write it out as many small files as you can
df.repartition(400).write.mode("overwrite").format("delta").saveAsTable("dbcertification_rev.demo_opt.transactions_slow")

# get number of files written
print("Files Written : ",spark.sql(""" DESCRIBE DETAIL dbcertification_rev.demo_opt.transactions_slow """).collect()[0]["numFiles"])

In [0]:
from time import time

t0 = time()

slow_df = spark.table("dbcertification_rev.demo_opt.transactions_slow")

result = (
    slow_df.groupBy("customer_id","region")
    .agg(sum("amount").alias("total_amount"), count("*").alias("txn_count"))
    .orderBy(col("total_amount").desc())
)

result.write.mode("overwrite").format("delta").saveAsTable("dbcertification_rev.demo_opt.slow_result")

print(f"Slow version took: {time() - t0:.1f} seconds")

In [0]:
# AQE is by default enabled in Serverless Compute

In [0]:
# run optimize to compact 400 small files into a handful of larger ones 
spark.sql(""" OPTIMIZE dbcertification_rev.demo_opt.transactions_slow ZORDER BY (customer_id)  """)

In [0]:
# same logic run again

from time import time

t0 = time()

slow_df = spark.table("dbcertification_rev.demo_opt.transactions_slow")

result = (
    slow_df.groupBy("customer_id","region")
    .agg(sum("amount").alias("total_amount"), count("*").alias("txn_count"))
    .orderBy(col("total_amount").desc())
)

result.write.mode("overwrite").format("delta").saveAsTable("dbcertification_rev.demo_opt.slow_result")

print(f"Slow version took: {time() - t0:.1f} seconds")